# 01 - EDA and Data Preparation

Notebook goals:
- Visualize the four baseline variables (`INDPRO`, `CPIAUCNS`, `FEDFUNDS`, `NASDAQCOM`)
- Run Stage-1 STL decomposition on daily NASDAQ data
- Show end-of-month sampling decision for frequency harmonization
- Run ADF tests for transformed monthly series

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from statsmodels.tsa.stattools import adfuller

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from nasdaq_svar.config import load_config
from nasdaq_svar.pipeline import run_pipeline
from nasdaq_svar.presentation import build_chapter1_conclusion
from nasdaq_svar.stage1_stl import run_stage1_stl
from nasdaq_svar.transforms import build_stage2_dataset, to_month_end

plt.style.use("seaborn-v0_8-whitegrid")
cfg = load_config(PROJECT_ROOT / "configs/default.yaml")
cfg["sample"]

In [ ]:
required_raw = [PROJECT_ROOT / "data/raw" / f"{k}.csv" for k in cfg["series"].keys()]
if not all(p.exists() for p in required_raw):
    _ = run_pipeline(config_path=PROJECT_ROOT / "configs/default.yaml", project_root=PROJECT_ROOT)

raw = {}
for alias, fred_id in cfg["series"].items():
    path = PROJECT_ROOT / "data/raw" / f"{alias}.csv"
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    raw[alias] = df.iloc[:, 0].rename(alias)

raw.keys()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=False)
for ax, key in zip(axes, ["INDPRO", "CPIAUCNS", "FEDFUNDS", "NASDAQCOM"]):
    raw[key].plot(ax=ax, lw=1.2)
    ax.set_title(key)
fig.tight_layout()

In [ ]:
stage1 = run_stage1_stl(raw["NASDAQCOM"], cfg["stage1"])
components = stage1.daily_components

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
components["observed_level"].plot(ax=axes[0], lw=1.0, title="Observed NASDAQ (Daily)")
components["trend"].plot(ax=axes[1], lw=1.0, title="STL Trend")
components["seasonal"].plot(ax=axes[2], lw=1.0, title="STL Seasonal")
components["resid"].plot(ax=axes[3], lw=0.8, title="STL Residual")
fig.tight_layout()

In [ ]:
monthly_sa_last = stage1.monthly_deseasonalized.rename("NASDAQ_SA_LAST")
monthly_sa_mean = components["deseasonalized_level"].resample("ME").mean().rename("NASDAQ_SA_MEAN")

compare = pd.concat([monthly_sa_last, monthly_sa_mean], axis=1).dropna()
compare[["NASDAQ_SA_LAST", "NASDAQ_SA_MEAN"]].tail(12)

In [ ]:
monthly_levels = pd.DataFrame(
    {
        "INDPRO": to_month_end(raw["INDPRO"]),
        "CPIAUCNS": to_month_end(raw["CPIAUCNS"]),
        "FEDFUNDS": to_month_end(raw["FEDFUNDS"]),
        "NASDAQ_SA": monthly_sa_last,
    }
).dropna()

stage2_data = build_stage2_dataset(
    monthly_levels,
    transforms=cfg["stage2"]["transforms"],
    ordering=cfg["stage2"]["ordering"],
)

def adf_report(series):
    stat, pvalue, usedlag, nobs, crit, _ = adfuller(series.dropna(), autolag="AIC")
    return {
        "adf_stat": stat,
        "pvalue": pvalue,
        "lag": usedlag,
        "nobs": nobs,
        "crit_5pct": crit["5%"],
    }

adf_table = pd.DataFrame({col: adf_report(stage2_data[col]) for col in stage2_data.columns}).T
adf_table

## Slide-Ready Conclusion

In [ ]:
chapter1_text = build_chapter1_conclusion(
    raw=raw,
    compare=compare,
    adf_table=adf_table,
    transforms=cfg["stage2"]["transforms"],
)
display(Markdown(chapter1_text))